# 4. SQL sorguları

`analytics/sql/` klasöründeki sorgular raporların ve sitenin altyapısı. Kullanılan teknikler: CTE, pencere fonksiyonları (`ROW_NUMBER`, `RANK`, `PERCENT_RANK`, `LAG`, kayan `AVG`/`MAX`), koşullu toplama ile pivot, alt sorgular.

In [1]:
import sys, sqlite3
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from mi import db, stats as st, events as ev, charts
%matplotlib inline
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
con = db.connect()
prices = pd.read_sql('SELECT * FROM prices', con)
close, volume = st.wide(prices), st.wide(prices, 'volume')
r = st.returns(close)
assets = pd.read_sql('SELECT * FROM assets', con).set_index('code')
print(f'{close.shape[1]} varlık, {close.index[0]:%Y-%m-%d} – {close.index[-1]:%Y-%m-%d}, {len(prices):,} fiyat satırı')

123 varlık, 2021-11-29 – 2026-11-27, 148,377 fiyat satırı


In [2]:
from IPython.display import Markdown
for f in sorted(db.SQL_DIR.glob('*.sql')):
    sql = f.read_text()
    display(Markdown(f'### {f.stem}\n```sql\n{sql}\n```'))
    display(db.query(f.stem, con).head(8))

### drawdowns
```sql
-- Her varlığın zirveden en büyük düşüşü (maksimum drawdown) ve şu an zirveden ne kadar uzakta olduğu.
-- Teknik: MAX() OVER (ROWS UNBOUNDED PRECEDING) ile o güne kadarki en yüksek kapanış.
WITH peak AS (
  SELECT code, date, close,
         MAX(close) OVER (PARTITION BY code ORDER BY date ROWS UNBOUNDED PRECEDING) AS run_max,
         ROW_NUMBER() OVER (PARTITION BY code ORDER BY date DESC) AS rn
  FROM prices
),
dd AS (SELECT code, date, rn, 100.0 * (close / run_max - 1) AS drawdown FROM peak)
SELECT d.code, a.name, a.cls,
       ROUND(MIN(drawdown), 1)                                AS max_drawdown,
       (SELECT date FROM dd x WHERE x.code = d.code ORDER BY drawdown LIMIT 1) AS trough_date,
       ROUND(MAX(CASE WHEN rn = 1 THEN drawdown END), 1)      AS current_drawdown
FROM dd d JOIN assets a USING (code)
GROUP BY d.code
ORDER BY max_drawdown;

```

,code,name,cls,max_drawdown,trough_date,current_drawdown
0,MRNA,Moderna,hisse,-93.90,2025-11-20,-38.00
1,APP,AppLovin,hisse,-90.40,2022-12-27,-45.60
2,MSTR,Strategy,hisse,-83.60,2026-11-24,-82.80
3,SHOP,Shopify,hisse,-83.60,2022-10-11,-20.90
4,PYPL,PayPal Holdings,hisse,-80.20,2026-02-12,-69.70
5,CRWV,CoreWeave,hisse,-78.50,2026-11-20,-76.00
6,RKLB,Rocket Lab,hisse,-77.70,2024-04-15,-66.10
7,DASH,DoorDash,hisse,-76.50,2022-10-14,-40.40


### fund_consensus
```sql
-- Takip edilen büyük fonların ortak tercihleri: kaç fon tutuyor, son çeyrekte kaçı artırdı/azalttı.
SELECT code,
       COUNT(*)                                                     AS funds,
       SUM(CASE WHEN change IN ('new', 'add') THEN 1 ELSE 0 END)    AS buyers,
       SUM(CASE WHEN change IN ('cut', 'out') THEN 1 ELSE 0 END)    AS sellers,
       ROUND(SUM(value_usd) / 1e9, 2)                               AS total_busd
FROM fund_holdings
WHERE code IS NOT NULL
GROUP BY code
HAVING funds >= 3
ORDER BY buyers - sellers DESC, funds DESC;

```

,code,funds,buyers,sellers,total_busd
0,MELI,5,5,0,1.08
1,MU,5,5,0,6.73
2,META,9,6,3,7.96
3,NVDA,7,5,2,6.36
4,INTC,5,4,1,4.44
5,CRWV,3,3,0,0.53
6,WDC,3,3,0,0.40
7,LRCX,6,4,2,4.69


### insider_by_sector
```sql
-- Son 180 günde yöneticilerin sektör bazında net alım/satımı ve plansız (10b5-1 dışı) satışların payı.
SELECT a.sector,
       COUNT(DISTINCT t.code)                                   AS companies,
       ROUND(SUM(t.buy_usd) / 1e6, 1)                           AS buy_musd,
       ROUND(SUM(t.sell_usd) / 1e6, 1)                          AS sell_musd,
       ROUND((SUM(t.buy_usd) - SUM(t.sell_usd)) / 1e6, 1)       AS net_musd,
       ROUND(100.0 * SUM(CASE WHEN t.plan = 0 THEN t.sell_usd ELSE 0 END) / NULLIF(SUM(t.sell_usd), 0), 0) AS unplanned_sell_pct
FROM insider_trades t JOIN assets a USING (code)
WHERE t.date >= date((SELECT MAX(date) FROM prices), '-180 day')
GROUP BY a.sector
ORDER BY net_musd DESC;

```

,sector,companies,buy_musd,sell_musd,net_musd,unplanned_sell_pct
0,Gıda ve içecek,4,23.90,33.20,-9.20,22.00
1,Otomotiv,2,0.00,9.60,-9.60,100.00
2,Ulaştırma,2,0.00,14.20,-14.20,100.00
3,Tüketici hizmetleri,2,0.30,136.10,-135.80,15.00
4,Sanayi,5,0.60,177.30,-176.70,57.00
5,Sağlık,10,0.50,294.10,-293.60,27.00
6,İletişim,6,0.00,345.50,-345.50,13.00
7,Havacılık ve uzay,4,1.00,420.10,-419.10,8.00


... (6 outputs left out)

### sector_returns
```sql
-- Sektörlerin eşit ağırlıklı getirisi ve sektör içi en iyi / en kötü hisse.
-- Teknik: CTE zinciri, pencere fonksiyonu ile sektör içi sıralama (RANK), GROUP BY ile toplulaştırma.
WITH ranked AS (
  SELECT code, date, close, ROW_NUMBER() OVER (PARTITION BY code ORDER BY date DESC) AS rn
  FROM prices
),
ret AS (
  SELECT code,
         100.0 * (MAX(CASE WHEN rn = 1 THEN close END) / MAX(CASE WHEN rn = 6 THEN close END) - 1)  AS r_1w,
         100.0 * (MAX(CASE WHEN rn = 1 THEN close END) / MAX(CASE WHEN rn = 22 THEN close END) - 1) AS r_1m,
         100.0 * (MAX(CASE WHEN rn = 1 THEN close END) / MAX(CASE WHEN rn = 64 THEN close END) - 1) AS r_3m
  FROM ranked WHERE rn <= 64 GROUP BY code
),
stock AS (
  SELECT a.sector, r.*,
         RANK() OVER (PARTITION BY a.sector ORDER BY r.r_1m DESC) AS best,
         RANK() OVER (PARTITION BY a.sector ORDER BY r.r_1m ASC)  AS worst
  FROM ret r JOIN assets a USING (code)
  WHERE a.cls = 'hisse' AND a.sector <> ''
)
SELECT sector,
       COUNT(*)                                  AS n,
       ROUND(AVG(r_1w), 2)                       AS r_1w,
       ROUND(AVG(r_1m), 2)                       AS r_1m,
       ROUND(AVG(r_3m), 2)                       AS r_3m,
       SUM(CASE WHEN r_1m > 0 THEN 1 ELSE 0 END) AS up_1m,
       MAX(CASE WHEN best = 1 THEN code END)     AS leader,
       MAX(CASE WHEN worst = 1 THEN code END)    AS laggard
FROM stock
GROUP BY sector
ORDER BY r_1m DESC;

```

,sector,n,r_1w,r_1m,r_3m,up_1m,leader,laggard
0,İletişim,6,5.21,6.69,-7.83,4,CMCSA,GOOGL
1,Perakende,10,4.25,6.20,4.23,7,COST,MELI
2,Sağlık,10,6.94,6.10,14.74,8,IDXX,ALNY
3,Seyahat ve eğlence,3,2.48,5.49,-3.99,3,ABNB,BKNG
4,Gıda ve içecek,5,2.02,4.21,4.96,4,MDLZ,MNST
5,Otomotiv,2,6.21,3.34,1.36,1,PCAR,TSLA
6,Tüketici hizmetleri,2,5.88,1.82,-9.30,1,PYPL,DASH
7,Sanayi,7,6.68,0.67,-2.52,4,FER,AXON


### trend_status
```sql
-- 50 ve 200 günlük hareketli ortalamalar ve fiyatın bunlara göre konumu (trend sağlığı).
-- Teknik: AVG() OVER (ROWS BETWEEN n PRECEDING AND CURRENT ROW) ile hareketli ortalama.
WITH ma AS (
  SELECT code, date, close,
         AVG(close) OVER (PARTITION BY code ORDER BY date ROWS BETWEEN 49 PRECEDING AND CURRENT ROW)  AS ma50,
         AVG(close) OVER (PARTITION BY code ORDER BY date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW) AS ma200,
         COUNT(*)   OVER (PARTITION BY code ORDER BY date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW) AS n,
         ROW_NUMBER() OVER (PARTITION BY code ORDER BY date DESC) AS rn
  FROM prices
)
SELECT code, date, close, ROUND(ma50, 2) AS ma50, ROUND(ma200, 2) AS ma200,
       ROUND(100.0 * (close / ma200 - 1), 1) AS dist_ma200,
       CASE WHEN close > ma50 AND ma50 > ma200 THEN 'güçlü yükseliş'
            WHEN close > ma200                  THEN 'yükseliş'
            WHEN close < ma50 AND ma50 < ma200  THEN 'güçlü düşüş'
            ELSE 'düşüş' END AS trend
FROM ma
WHERE rn = 1 AND n = 200;

```

,code,date,close,ma50,ma200,dist_ma200,trend
0,AAPL,2026-11-27,358.54,343.27,304.38,17.80,güçlü yükseliş
1,ABNB,2026-11-27,179.16,170.18,151.49,18.30,güçlü yükseliş
2,ADBE,2026-11-27,234.79,241.68,245.70,-4.40,güçlü düşüş
3,ADI,2026-11-27,346.00,394.11,377.22,-8.30,düşüş
4,ADP,2026-11-27,319.43,288.06,244.28,30.80,güçlü yükseliş
5,ADSK,2026-11-27,251.88,240.53,234.23,7.50,güçlü yükseliş
6,AEP,2026-11-27,127.67,124.52,127.10,0.40,yükseliş
7,ALAB,2026-11-27,206.84,279.97,259.46,-20.30,düşüş
